# SFT Training: LLaMA 3-8B-Instruct (Glyph Reasoning)

Fine-tune `meta-llama/Meta-Llama-3-8B-Instruct` on the glyph reasoning dataset using LoRA.  
Based on `train/train_sft.py` (Qwen2.5-7B pipeline), adapted for LLaMA 3 architecture.

**Requirements:** GPU with >= 24GB VRAM (A100/A10G/L4). Uses LoRA + gradient checkpointing to fit in memory.

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install torch transformers datasets peft accelerate bitsandbytes tqdm
# Flash Attention (optional but recommended — speeds up training ~30%, requires CUDA):
# !pip install flash-attn --no-build-isolation

## 1. Configuration

In [ ]:
import os

# --- HuggingFace auth (LLaMA 3 is a gated model) ---
# Set HF_TOKEN env var or run: huggingface-cli login
assert os.environ.get("HF_TOKEN") or os.path.exists(os.path.expanduser("~/.cache/huggingface/token")), \
    "LLaMA 3 is gated. Set HF_TOKEN env var or run: huggingface-cli login"

# --- Model & Data ---
MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
DATA_FILE = "data/sft_final.jsonl"
OUTPUT_DIR = os.environ.get(
    "OUTPUT_DIR",
    "checkpoints/llama3-8b-glyph-sft",
)

# --- LoRA hyperparameters ---
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# --- Training hyperparameters ---
MAX_SEQ_LENGTH = 2048
BATCH_SIZE = 1
GRAD_ACC_STEPS = 16
LEARNING_RATE = 2e-4
NUM_EPOCHS = 1
WARMUP_STEPS = 20

# --- Cache (for cloud environments like RunPod/Modal) ---
if os.path.exists("/workspace"):
    os.environ.setdefault("HF_HOME", "/workspace/huggingface_cache")
    os.environ.setdefault("HF_DATASETS_CACHE", "/workspace/huggingface_cache/datasets")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Model:      {MODEL_NAME}")
print(f"Data:       {DATA_FILE}")
print(f"Output:     {OUTPUT_DIR}")

## 2. Load Tokenizer & Model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# LLaMA 3 uses <|eot_id|> as EOS but has no default pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Use flash_attention_2 if available, fall back to sdpa
attn_impl = "eager"
try:
    import flash_attn
    attn_impl = "flash_attention_2"
    print("Using Flash Attention 2")
except ImportError:
    attn_impl = "sdpa"
    print("flash-attn not installed, using SDPA (still efficient)")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    attn_implementation=attn_impl,
)

print(f"Model loaded: {model.config._name_or_path}")
print(f"Parameters:   {model.num_parameters() / 1e9:.1f}B")

## 3. Apply LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
)

model = get_peft_model(model, peft_config)
# Required for LoRA + gradient checkpointing compatibility
model.enable_input_require_grads()
model.print_trainable_parameters()

## 4. Prepare Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files=DATA_FILE, split="train")
print(f"Raw examples: {len(dataset)}")

def tokenize(example):
    """Apply LLaMA 3 chat template and tokenize."""
    if not example.get("messages") or len(example["messages"]) == 0:
        return {"input_ids": [], "labels": []}
    try:
        text = tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
        )
        inputs = tokenizer(
            text,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
        )
        inputs["labels"] = inputs["input_ids"].copy()
        return inputs
    except Exception as e:
        print(f"Skipping example due to error: {e}")
        return {"input_ids": [], "labels": []}

dataset = dataset.map(tokenize, remove_columns=dataset.column_names)
dataset = dataset.filter(lambda x: len(x["input_ids"]) > 0)
print(f"Tokenized examples: {len(dataset)}")

# Preview sequence length distribution
lengths = [len(x["input_ids"]) for x in dataset]
print(f"Seq lengths — min: {min(lengths)}, max: {max(lengths)}, avg: {sum(lengths)/len(lengths):.0f}")

## 5. Training

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
from transformers.trainer_utils import get_last_checkpoint

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    bf16=True,
    fp16=False,
    logging_steps=5,
    logging_first_step=True,
    save_steps=100,
    save_total_limit=1,
    report_to="none",
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    gradient_checkpointing=True,
    group_by_length=True,
    ddp_find_unused_parameters=False,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, pad_to_multiple_of=8),
)

# Resume from checkpoint if one exists
last_checkpoint = get_last_checkpoint(OUTPUT_DIR)
if last_checkpoint:
    print(f"Resuming from checkpoint: {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Starting training from scratch...")
    trainer.train()

## 6. Save Model

In [ ]:
# Save LoRA adapter weights
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

## 7. Quick Inference Test

Load the trained adapter and verify glyph emergence on a sample prompt.

In [ ]:
from peft import PeftModel

# Reload base model for inference
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
# Load trained LoRA adapter
infer_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
infer_model.eval()

device = infer_model.device
print(f"Inference model loaded on {device}")


def generate(prompt, max_new_tokens=512):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = infer_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Strip input tokens from output
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

In [ ]:
# Test with a latent prompt — expect glyph symbols to emerge autonomously
test_prompt = """
Solve the following problem carefully.
Do not mention any tags, symbols, or special formatting.

Problem:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
"""

response = generate(test_prompt)
print("Response:\n" + "-" * 40)
print(response)

In [ ]:
# Try your own prompt
my_prompt = """
Solve the following problem carefully.
Do not mention any tags, symbols, or special formatting.

Problem:
There are 5 houses on a street. Each house has 3 windows. If 4 windows are broken in total, how many unbroken windows are there?
"""

print(generate(my_prompt))

## 8. Evaluation: Base vs Fine-tuned

Compare `Meta-Llama-3-8B-Instruct` (base) against the LoRA fine-tuned model across three prompting strategies: **Glyph**, **XML**, and **Natural**.

Metrics: accuracy, structure violation rate, avg reasoning/answer tokens.

**Note:** This section reloads models from scratch — previous inference models are freed first.

In [ ]:
import json
import re
import gc
import random
from tqdm import tqdm

# Free VRAM from inference test (section 7) before loading eval models
for var in ["infer_model", "base_model"]:
    if var in dir():
        exec(f"del {var}")
gc.collect()
torch.cuda.empty_cache()

# --- Eval config ---
EVAL_DATA = "data/unified_dataset.jsonl"
EVAL_LIMIT = 20  # set to None for full eval
EVAL_SEED = 42
MAX_NEW_TOKENS = 512

# Models to compare: base LLaMA 3 vs fine-tuned LoRA adapter
EVAL_MODELS = [
    MODEL_NAME,   # base: meta-llama/Meta-Llama-3-8B-Instruct
    OUTPUT_DIR,   # fine-tuned: checkpoints/llama3-8b-glyph-sft
]

# --- Prompt templates (from eval/prompts.py) ---
def xml_prompt(q):
    return f"""
You must solve the problem using the structure below.

<guideline>
Solve the problem carefully.
<plan>
Describe the approach.
</plan>
<step>
Work through the calculation.
</step>
<takeaway>
Give the final numeric answer.
</takeaway>
</guideline>

Problem:
{q}
"""

def natural_prompt(q):
    return f"""
Solve the problem using this structure:

Guideline:
Plan:
Step:
Takeaway:

Problem:
{q}
"""

def glyph_prompt(q):
    return f"""
Solve the problem using the glyph structure:

\U0001f71e Solve the problem carefully.
\U0001f706 Describe the approach.
\U0001f702 Work through the calculation.
\U0001f703 Give the final numeric answer.

Problem:
{q}
"""

PROMPTS = {
    "glyph": glyph_prompt,
    "xml": xml_prompt,
    "natural": natural_prompt,
}

# --- Helper functions (from eval/eval_structures.py) ---
def extract_answer(text):
    nums = re.findall(r"-?\d+\.?\d*", text)
    return nums[-1] if nums else None

def structure_violation(text, mode):
    if mode == "xml":
        return not all(tag in text for tag in ["<guideline>", "<plan>", "<step>", "<takeaway>"])
    if mode == "natural":
        return not all(k in text for k in ["Guideline", "Plan", "Step", "Takeaway"])
    if mode == "glyph":
        return not all(g in text for g in ["\U0001f71e", "\U0001f706", "\U0001f702", "\U0001f703"])
    return True

# --- Load eval tasks (JSONL, shuffled) ---
eval_tasks = []
with open(EVAL_DATA, "r") as f:
    for line in f:
        if line.strip():
            eval_tasks.append(json.loads(line))

random.seed(EVAL_SEED)
random.shuffle(eval_tasks)
if EVAL_LIMIT:
    eval_tasks = eval_tasks[:EVAL_LIMIT]
print(f"Eval tasks loaded: {len(eval_tasks)} (shuffled, seed={EVAL_SEED})")

In [ ]:
def evaluate_model(model_name_or_path, tasks):
    """Evaluate a model (HF hub ID or local adapter path) across all prompt modes."""
    from peft import PeftModel, PeftConfig

    print(f"\n{'='*50}")
    print(f"Evaluating: {model_name_or_path}")
    print(f"{'='*50}")

    # Detect if this is a LoRA adapter (local dir with adapter_config.json)
    is_adapter = os.path.exists(os.path.join(model_name_or_path, "adapter_config.json"))

    if is_adapter:
        print("Detected LoRA adapter, loading base + adapter...")
        config = PeftConfig.from_pretrained(model_name_or_path)
        base_model = AutoModelForCausalLM.from_pretrained(
            config.base_model_name_or_path,
            torch_dtype=torch.bfloat16,
            device_map="auto",
        )
        eval_model = PeftModel.from_pretrained(base_model, model_name_or_path)
        eval_tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
    else:
        print("Loading full model from hub...")
        eval_tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
        eval_model = AutoModelForCausalLM.from_pretrained(
            model_name_or_path,
            torch_dtype=torch.bfloat16,
            device_map="auto",
        )

    eval_model.eval()
    dev = next(eval_model.parameters()).device

    marker_map = {"glyph": "\U0001f703", "xml": "<takeaway>", "natural": "Takeaway:"}
    results = {}

    for mode, prompt_fn in PROMPTS.items():
        correct = 0
        violations = 0
        total_reasoning_tokens = 0
        total_answer_tokens = 0

        marker = marker_map.get(mode)

        for task in tqdm(tasks, desc=f"[{mode}]"):
            raw_prompt = prompt_fn(task["question"])
            messages = [{"role": "user", "content": raw_prompt}]
            try:
                text_input = eval_tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )
            except Exception:
                text_input = raw_prompt

            inputs = eval_tokenizer(text_input, return_tensors="pt").to(dev)
            input_len = inputs["input_ids"].shape[1]

            with torch.no_grad():
                output = eval_model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    pad_token_id=eval_tokenizer.eos_token_id,
                )

            gen_text = eval_tokenizer.decode(output[0][input_len:], skip_special_tokens=True)

            # Split reasoning vs answer by marker
            reasoning_part, answer_part = gen_text, ""
            if marker and marker in gen_text:
                parts = gen_text.split(marker)
                answer_part = parts[-1]
                reasoning_part = gen_text[:-(len(answer_part) + len(marker))]

            search_text = answer_part if (marker and marker in gen_text) else gen_text
            extracted = extract_answer(search_text)
            if extracted == task["answer"]:
                correct += 1
            if structure_violation(gen_text, mode):
                violations += 1

            total_reasoning_tokens += len(eval_tokenizer.encode(reasoning_part, add_special_tokens=False))
            total_answer_tokens += len(eval_tokenizer.encode(answer_part, add_special_tokens=False))

        n = len(tasks)
        results[mode] = {
            "accuracy": correct / n,
            "violation_rate": violations / n,
            "avg_reasoning_tok": total_reasoning_tokens / n,
            "avg_answer_tok": total_answer_tokens / n,
            "avg_total_tok": (total_reasoning_tokens + total_answer_tokens) / n,
        }

    # Free VRAM
    del eval_model
    if is_adapter:
        del base_model
    del eval_tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    return results

In [ ]:
# Run evaluation on both models
all_results = {}
for model_id in EVAL_MODELS:
    result = evaluate_model(model_id, eval_tasks)
    if result:
        all_results[model_id] = result

# Print results
print("\n" + "=" * 70)
print("EVALUATION RESULTS")
print("=" * 70)
for model_id, modes in all_results.items():
    print(f"\nModel: {model_id}")
    print(f"{'Mode':<10} {'Acc':>8} {'Violation':>10} {'Reason Tok':>12} {'Answer Tok':>12} {'Total Tok':>12}")
    print("-" * 64)
    for mode, stats in modes.items():
        print(
            f"{mode:<10} {stats['accuracy']:>8.1%} {stats['violation_rate']:>10.1%} "
            f"{stats['avg_reasoning_tok']:>12.1f} {stats['avg_answer_tok']:>12.1f} "
            f"{stats['avg_total_tok']:>12.1f}"
        )

In [ ]:
# Save results to CSV
import csv

eval_output_csv = "eval/eval_results_llama3_sft.csv"
if all_results:
    with open(eval_output_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "Model", "Mode", "Accuracy", "Structure Violation Rate",
            "Avg Reasoning Tokens", "Avg Answer Tokens", "Avg Total Tokens",
        ])
        for model_id, modes in all_results.items():
            for mode, stats in modes.items():
                writer.writerow([
                    model_id, mode,
                    f"{stats['accuracy']:.4f}",
                    f"{stats['violation_rate']:.4f}",
                    f"{stats['avg_reasoning_tok']:.2f}",
                    f"{stats['avg_answer_tok']:.2f}",
                    f"{stats['avg_total_tok']:.2f}",
                ])
    print(f"Results saved to {eval_output_csv}")